# MeterMind — Model 1: Rule-Based DP Reorderer

This notebook implements the **rule-based baseline** for MeterMind.

Given an expanded archaic prose line (e.g. `Thou feedest thy light's own flame with fuel that is self-substantial`), the DP reorderer finds the word ordering that **best matches the iambic pentameter stress pattern** — purely through symbolic constraint-solving, no learning involved.

### Why DP and not greedy?
A greedy approach fills each stress position with the first matching word it finds. This can paint itself into a corner — grabbing a word early that would have been more useful later. Dynamic programming instead considers **all possible orderings** and finds the globally optimal one.

### Pipeline
```
training_pairs.csv → stress extraction (CMU) → DP reorder → evaluate (MA, SP, G)
```

## 0. Install dependencies

In [ ]:
%pip install pronouncing sentence-transformers --quiet

## 1. Imports

In [ ]:
import re
import io
import math
import pronouncing
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from sentence_transformers import SentenceTransformer, util

print('Imports ready.')

## 2. Load dataset

In [ ]:
# Upload training_pairs.csv when prompted
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_csv(io.StringIO(uploaded[filename].decode('utf-8')))

# Column names from dataset_generation.ipynb are 'input' and 'target'
pairs = list(zip(df['input'].tolist(), df['target'].tolist()))

print(f'Loaded {len(pairs):,} pairs')
print(f'\nExample:')
print(f'  INPUT  : {pairs[0][0]}')
print(f'  TARGET : {pairs[0][1]}')

## 3. Stress extraction

Maps each word to its syllable stress pattern using the CMU Pronouncing Dictionary.

- **Multisyllabic words** (e.g. `compare` → `[0,1]`) have a fixed stress pattern.
- **Monosyllabic words marked as stressed** (e.g. `shall`, `thee`, `day`) are treated as **flexible** — they can sit in either a stressed or unstressed position. This reflects how poets naturally use function words.

In [ ]:
# ── Shared metric utilities ───────────────────────────────────────────────────
# Upload meter_utils.py when prompted, then eval_metrics.py, then imports tokenize, get_stress,
# n_syllables, is_flexible, metrical_accuracy, semantic_preservation,
# grammaticality, load_sp_model, load_gpt2, and IAMBIC_TEMPLATE.
from google.colab import files as _fu
print('Please upload meter_utils.py')
_fu.upload()
from meter_utils import *
print('Please upload eval_metrics.py')
_fu.upload()
from eval_metrics import *

# Sanity check
test_words = ['shall', 'i', 'compare', 'thee', 'to', 'a', 'summers', 'day', 'feedest', 'flame']
print(f"{'word':12} syllables  stress      flexible")
print("-" * 45)
for w in test_words:
    print(f"{w:12} {n_syllables(w):<10} {str(get_stress(w)):12} {is_flexible(w)}")


## 4. DP Reorderer

### How it works

Iambic pentameter has 10 syllable positions: `[0,1,0,1,0,1,0,1,0,1]` (unstressed, stressed, alternating).

The DP solves this as an **assignment problem**:
- **State:** `(syllable_position, set_of_words_used_so_far)`
- **Transition:** try placing each unused word next; score how well its stress pattern matches the template at the current position
- **Goal:** find the word ordering that maximises total stress matches

Because the number of words per line is small (~10-20), we use **bitmask DP** — each word is represented as a bit in an integer, making state representation compact and fast.

### Scoring
Each syllable earns 1 point if it matches the iambic template. Flexible monosyllables earn 1 point in any position. The DP maximises total points across all 10 template positions.

In [ ]:
IAMBIC_TEMPLATE = [0, 1, 0, 1, 0, 1, 0, 1, 0, 1]   # 10 positions: unstressed-STRESSED x5
MAX_TEMPLATE_LEN = len(IAMBIC_TEMPLATE)


def word_syllable_score(word, start_pos):
    """
    Score how well `word` fits into the iambic template starting at `start_pos`.
    Returns (score, syllables_consumed).
    """
    stress = get_stress(word)
    score  = 0

    if is_flexible(word):
        # Flexible monosyllable: scores 1 only within template, but always consumes 1 syllable
        score = 1 if start_pos < MAX_TEMPLATE_LEN else 0
        syllables_consumed = 1
    else:
        syllables_consumed = 0
        for i, s in enumerate(stress):
            pos = start_pos + i
            if pos < MAX_TEMPLATE_LEN:
                if s == IAMBIC_TEMPLATE[pos]:
                    score += 1
            syllables_consumed += 1   # always consume, even past template end

    return score, max(syllables_consumed, 1)


def dp_reorder(words):
    """
    Find the word ordering that maximises iambic stress match using bitmask DP.

    State: (syllable_position, bitmask_of_used_words)
    Value: best score achievable from this state

    Returns the reordered word list.
    """
    # Cap at 20 words to keep DP tractable (2^20 = 1M states)
    # Words beyond 20 are appended at the end unchanged
    cap      = min(len(words), 20)
    core     = words[:cap]
    leftover = words[cap:]
    n        = len(core)

    # Pre-compute scores for each (word, start_position) pair
    # score_cache[i][pos] = (score, syllables_consumed) for word i at template position pos
    score_cache = {}
    for i, w in enumerate(core):
        for pos in range(MAX_TEMPLATE_LEN + 1):
            score_cache[(i, pos)] = word_syllable_score(w, pos)

    # DP table: dp[mask][pos] = best score with `mask` words used, at syllable position `pos`
    FULL_MASK = (1 << n) - 1
    NEG_INF   = float('-inf')

    # Use dicts for sparse storage
    dp     = {}   # (mask, pos) -> best_score
    parent = {}   # (mask, pos) -> (prev_mask, prev_pos, word_idx)

    dp[(0, 0)] = 0

    # Process states in order of number of bits set (words placed)
    for num_placed in range(n):
        for (mask, pos), score in list(dp.items()):
            if bin(mask).count('1') != num_placed:
                continue
            # Try placing each unused word
            for i in range(n):
                if mask & (1 << i):
                    continue   # already used

                word_score, syl_consumed = score_cache[(i, pos)]
                new_mask  = mask | (1 << i)
                new_pos   = min(pos + syl_consumed, MAX_TEMPLATE_LEN)
                new_score = score + word_score

                if dp.get((new_mask, new_pos), NEG_INF) < new_score:
                    dp[(new_mask, new_pos)]     = new_score
                    parent[(new_mask, new_pos)] = (mask, pos, i)

    # Find best final state (all words placed)
    best_score = NEG_INF
    best_state = None
    for pos in range(MAX_TEMPLATE_LEN + 1):
        s = dp.get((FULL_MASK, pos), NEG_INF)
        if s > best_score:
            best_score = s
            best_state = (FULL_MASK, pos)

    # Backtrack to recover word order
    if best_state is None or best_state not in parent:
        return words   # fallback: return original order

    order = []
    state = best_state
    while state in parent:
        prev_mask, prev_pos, word_idx = parent[state]
        order.append(word_idx)
        state = (prev_mask, prev_pos)
    order.reverse()

    reordered = [core[i] for i in order] + leftover
    return reordered


# Quick test
test_line = "thou feedest thy light own flame with fuel that is self substantial"
words     = tokenize(test_line)
result    = dp_reorder(words)
print('Input    :', ' '.join(words))
print('Reordered:', ' '.join(result))

### 4a. Sample DP reorderings

Upload your `training_pairs.csv` here to sanity-check the reorderer on a handful of real pairs before running the full pipeline.

In [ ]:
# ── Sample DP reorderings on a few real pairs ─────────────────────────────────
import io
from google.colab import files as _files

_uploaded = _files.upload()
_filename = list(_uploaded.keys())[0]
_preview_df = pd.read_csv(io.StringIO(_uploaded[_filename].decode('utf-8')))

N_PREVIEW = 5   # change to see more rows

print(f"Showing {N_PREVIEW} sample reorderings\n")
print(f"Template : {IAMBIC_TEMPLATE}")
print(f"{'─'*70}")

for i, (prose_input, target) in enumerate(_preview_df[['input', 'target']].itertuples(index=False)):
    if i >= N_PREVIEW:
        break

    words     = tokenize(prose_input)
    reordered = dp_reorder(words)
    output    = ' '.join(reordered)

    # Quick inline MA (metrics cell may not be run yet)
    correct, syl_pos = 0, 0
    for word in reordered:
        if syl_pos >= len(IAMBIC_TEMPLATE): break
        if is_flexible(word):
            correct += 1; syl_pos += 1
        else:
            for s in get_stress(word):
                if syl_pos >= len(IAMBIC_TEMPLATE): break
                if s == IAMBIC_TEMPLATE[syl_pos]: correct += 1
                syl_pos += 1
    ma = correct / len(IAMBIC_TEMPLATE)

    changed = '(reordered)' if words != reordered else '(unchanged)'
    print(f"[{i+1}] {changed}")
    print(f"  INPUT    : {prose_input}")
    print(f"  TARGET   : {target}")
    print(f"  OUTPUT   : {output}")
    print(f"  MA       : {ma:.3f}")
    print()


## 5. Evaluation metrics

Three metrics, matching the MeterMind evaluation framework:

| Metric | What it measures | How |
|--------|-----------------|-----|
| **MA** — Metrical Accuracy | How well the output matches iambic stress pattern | CMU stress vs template |
| **SP** — Semantic Preservation | How much meaning is retained | Cosine similarity of SentenceTransformer embeddings |
| **G** — Grammaticality | How fluent/natural the output reads | GPT-2 perplexity (lower = better) |

In [ ]:
# ── Load models (SentenceTransformer + GPT-2) ────────────────────────────────
# meter_utils lazy-loads these on first call, but we trigger them here so the
# download happens before the eval loop rather than mid-run.
load_sp_model()
load_gpt2()
print('All metrics ready.')


### 5a. Sanity-check: Metrical Accuracy on uploaded sample pairs

Upload your `training_pairs.csv` here to spot-check `metrical_accuracy` on a handful of rows — before running the full pipeline. We print the per-syllable alignment so you can see exactly where matches and misses fall.

In [ ]:
# ── Upload a sample of training_pairs.csv and check MA ────────────────────────
import io
from google.colab import files as colab_files

sample_uploaded = colab_files.upload()
sample_filename = list(sample_uploaded.keys())[0]
sample_df = pd.read_csv(io.StringIO(sample_uploaded[sample_filename].decode('utf-8')))

N_SAMPLE = 10   # how many pairs to inspect; change as needed
sample_pairs = list(zip(sample_df['input'], sample_df['target']))[:N_SAMPLE]

def ma_debug(words, template=IAMBIC_TEMPLATE):
    """
    Like metrical_accuracy() but prints a per-syllable alignment table
    so you can verify the logic visually.
    """
    correct = 0
    syl_pos = 0
    rows = []

    for word in words:
        if syl_pos >= len(template):
            rows.append((word, '-', '-', '-', 'OVERFLOW'))
            continue
        if is_flexible(word):
            rows.append((word, 'flex', template[syl_pos], '✓', syl_pos))
            correct += 1
            syl_pos += 1
        else:
            stress = get_stress(word)
            for s in stress:
                if syl_pos >= len(template):
                    break
                match = s == template[syl_pos]
                rows.append((word, s, template[syl_pos], '✓' if match else '✗', syl_pos))
                if match:
                    correct += 1
                syl_pos += 1

    ma = correct / len(template)
    return ma, rows

print(f"Checking {N_SAMPLE} pairs — template: {IAMBIC_TEMPLATE}\n")
print(f"{'#':<3} {'MA':>5}  Input line")
print('-' * 60)

for idx, (prose_input, target) in enumerate(sample_pairs):
    words = tokenize(prose_input)
    ma, rows = ma_debug(words)
    print(f"\n[{idx+1}] MA={ma:.3f}  input : {prose_input}")
    print(f"{'':7}        target: {target}")
    print(f"  {'word':<14} {'stress':>6}  {'tmpl':>4}  {'match':>5}  pos")
    print(f"  {'-'*14} {'-'*6}  {'-'*4}  {'-'*5}  ---")
    for word, stress, tmpl, match, pos in rows:
        print(f"  {word:<14} {str(stress):>6}  {str(tmpl):>4}  {match:>5}  {pos}")


## 6. Run DP reorderer on full dataset

In [ ]:
from tqdm.notebook import tqdm

results = []

for prose_input, target in tqdm(pairs, desc='Reordering'):
    words    = tokenize(prose_input)
    reordered = dp_reorder(words)
    output   = ' '.join(reordered)

    ma = metrical_accuracy(reordered)
    sp = semantic_preservation(' '.join(tokenize(prose_input)), output)
    g  = grammaticality(output)

    results.append({
        'input':    prose_input,
        'target':   target,
        'output':   output,
        'MA':       round(ma, 4),
        'SP':       round(sp, 4),
        'G':        round(g,  4),
    })

results_df = pd.DataFrame(results)
print(f'Done. {len(results_df):,} results.')

## 7. Results

In [ ]:
print('=== DP Reorderer — Aggregate Results ===')
print(f"Metrical Accuracy (MA) : {results_df['MA'].mean():.4f} ± {results_df['MA'].std():.4f}")
print(f"Semantic Preservation (SP): {results_df['SP'].mean():.4f} ± {results_df['SP'].std():.4f}")
print(f"Grammaticality (G)     : {results_df['G'].mean():.4f} ± {results_df['G'].std():.4f}")

print('\n=== Sample outputs ===')
for _, row in results_df.head(5).iterrows():
    print(f"INPUT  : {row['input']}")
    print(f"TARGET : {row['target']}")
    print(f"OUTPUT : {row['output']}")
    print(f"MA={row['MA']:.3f}  SP={row['SP']:.3f}  G={row['G']:.3f}")
    print()

In [ ]:
# Distribution plots
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('DP Reorderer — Metric Distributions', fontsize=14)

for ax, metric, colour in zip(axes, ['MA', 'SP', 'G'], ['steelblue', 'seagreen', 'tomato']):
    ax.hist(results_df[metric], bins=30, color=colour, edgecolor='white', alpha=0.85)
    ax.axvline(results_df[metric].mean(), color='black', linestyle='--', linewidth=1.5, label=f"mean={results_df[metric].mean():.3f}")
    ax.set_title(metric)
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')
    ax.legend()

plt.tight_layout()
plt.savefig('dp_reorderer_metrics.png', dpi=150)
plt.show()

## 8. Save results

In [ ]:
results_df.to_csv('dp_reorderer_results.csv', index=False)
print('Saved dp_reorderer_results.csv')

# Summary dict — import this into the evaluation notebook to compare all 3 models
import json
summary = {
    'model': 'DP Reorderer',
    'n':     len(results_df),
    'MA':    {'mean': round(float(results_df['MA'].mean()), 4), 'std': round(float(results_df['MA'].std()), 4)},
    'SP':    {'mean': round(float(results_df['SP'].mean()), 4), 'std': round(float(results_df['SP'].std()), 4)},
    'G':     {'mean': round(float(results_df['G'].mean()),  4), 'std': round(float(results_df['G'].std()),  4)},
}
with open('dp_reorderer_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved dp_reorderer_summary.json')

# Download both
files.download('dp_reorderer_results.csv')
files.download('dp_reorderer_summary.json')
files.download('dp_reorderer_metrics.png')